## Limpieza de datos para entrenamiento. 

In [ ]:
"""
limpieza de datos de anestesia 
Basado en el artículo: "A Transformer-based Prediction Method for Depth of Anesthesia."

Fases:
  1. Validación e interpolación de señales crudas
  2. Re-muestreo a bloques de 5 segundos (tasa de droga y promedio BIS)
  3. Suavizado LOWESS sobre la señal BIS
  4. Normalización de covariables estáticas (edad, peso, altura, sexo)
  5. Guardado de archivos limpios en Entrenamiento_Limpios/
"""

import os
import re
import numpy as np
import pandas as pd
from statsmodels.nonparametric.smoothers_lowess import lowess

# ─────────────────────────────────────────────
#  RUTAS
# ─────────────────────────────────────────────
CARPETA_ENTRADA  = r"C:\Users\yordi\Desktop\Entrenamiento_Crudos"
CARPETA_SALIDA   = r"C:\Users\yordi\Desktop\Entrenamiento_Limpios"

# ─────────────────────────────────────────────
#  PARÁMETROS DE NORMALIZACIÓN (FASE 4)
# ─────────────────────────────────────────────
EDAD_MIN   = 16.00;   EDAD_MAX   = 90.00
PESO_MIN   = 34.10;   PESO_MAX   = 112.80
ALTURA_MIN = 140.20;  ALTURA_MAX = 186.60

# ─────────────────────────────────────────────
#  PARÁMETROS GENERALES
# ─────────────────────────────────────────────
TAMANO_BLOQUE   = 10          # segundos por bloque (Fase 2)
LOWESS_FRAC     = 0.03       # fracción de suavizado LOWESS (Fase 3)
BIS_MINIMO      = 20         # valores de BIS por debajo de este umbral → NaN (Fase 1)


# ══════════════════════════════════════════════════════════════════════════════
# FASE 1 – Validación e interpolación lineal
# ══════════════════════════════════════════════════════════════════════════════
def fase1_validacion_interpolacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    - BIS < 20  → NaN
    - PPF20_VOL y RFTN20_VOL: si un valor es menor que el anterior → NaN
      (se aplica de forma iterativa para cubrir caídas consecutivas)
    - Interpolación lineal sobre todos los NaN de las tres señales
    """
    df = df.copy()

    # --- BIS: marcar valores por debajo del mínimo clínico ---
    mascara_bis_invalido = df["BIS"] < BIS_MINIMO
    df.loc[mascara_bis_invalido, "BIS"] = np.nan
    n_bis = mascara_bis_invalido.sum()
    if n_bis:
        print(f"    [Fase 1] BIS: {n_bis} valor(es) < {BIS_MINIMO} marcados como NaN")

    # --- Volúmenes acumulados: deben ser no-decrecientes ---
    for columna_vol in ["PPF20_VOL", "RFTN20_VOL"]:
        serie = df[columna_vol].copy()
        n_invalidos = 0

        # Recorremos desde el segundo elemento
        for i in range(1, len(serie)):
            val_actual   = serie.iloc[i]
            val_anterior = serie.iloc[i - 1]

            # Si el anterior ya es NaN, comparamos con el último válido conocido
            if pd.isna(val_anterior):
                # Buscamos hacia atrás el último valor válido
                j = i - 1
                while j >= 0 and pd.isna(serie.iloc[j]):
                    j -= 1
                if j < 0:
                    continue           # no hay referencia válida, dejamos el valor
                val_anterior = serie.iloc[j]

            if not pd.isna(val_actual) and val_actual < val_anterior:
                serie.iloc[i] = np.nan
                n_invalidos += 1

        df[columna_vol] = serie
        if n_invalidos:
            print(f"    [Fase 1] {columna_vol}: {n_invalidos} valor(es) decreciente(s) marcados como NaN")

    # --- Interpolación lineal para rellenar todos los NaN ---
    for col in ["BIS", "PPF20_VOL", "RFTN20_VOL"]:
        n_nan_antes = df[col].isna().sum()
        df[col] = df[col].interpolate(method="linear", limit_direction="both")
        n_nan_despues = df[col].isna().sum()
        if n_nan_antes:
            print(f"    [Fase 1] {col}: {n_nan_antes} NaN interpolados "
                  f"({n_nan_despues} restantes tras interpolación)")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# FASE 2 – Re-muestreo a bloques de 5 s
# ══════════════════════════════════════════════════════════════════════════════
def fase2_remuestreo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa las muestras en bloques de TAMANO_BLOQUE segundos.

    - PPF20_RATE  = (vol_final_bloque − vol_inicio_bloque) / TAMANO_BLOQUE
    - RFTN20_RATE = ídem para remifentanilo
    - BIS         = promedio del bloque

    Se preserva la sincronización temporal; 
    El último bloque puede tener menos puntos (se procesa igual).
    """
    filas_resultado = []

    # Identificamos cuántas filas tiene el dataframe
    n_filas = len(df)
    inicio  = 0

    while inicio < n_filas:
        fin = min(inicio + TAMANO_BLOQUE, n_filas)   # índice exclusivo
        bloque = df.iloc[inicio:fin]

        # Tiempo del bloque (múltiplo de TAMANO_BLOQUE)
        t_bloque = (inicio // TAMANO_BLOQUE) * TAMANO_BLOQUE

        # Tasa = diferencia de volumen acumulado / duración real del bloque
        duracion_real = fin - inicio   # número de segundos reales en el bloque

        vol_inicio_ppf  = bloque["PPF20_VOL"].iloc[0]
        vol_fin_ppf     = bloque["PPF20_VOL"].iloc[-1]
        tasa_ppf        = (vol_fin_ppf - vol_inicio_ppf) / duracion_real

        vol_inicio_rftn = bloque["RFTN20_VOL"].iloc[0]
        vol_fin_rftn    = bloque["RFTN20_VOL"].iloc[-1]
        tasa_rftn       = (vol_fin_rftn - vol_inicio_rftn) / duracion_real

        # Promedio de BIS en el bloque
        bis_promedio = bloque["BIS"].mean()

        # Tomamos las covariables estáticas del primer punto (son constantes)
        edad   = bloque["Edad"].iloc[0]
        peso   = bloque["Peso"].iloc[0]
        altura = bloque["Altura"].iloc[0]
        sexo   = bloque["Sexo"].iloc[0]

        filas_resultado.append({
            "t"           : t_bloque,
            "PPF20_RATE"  : tasa_ppf,
            "RFTN20_RATE" : tasa_rftn,
            "BIS"         : bis_promedio,
            "Edad"        : edad,
            "Peso"        : peso,
            "Altura"      : altura,
            "Sexo"        : sexo,
        })

        inicio += TAMANO_BLOQUE

    df_resampleado = pd.DataFrame(filas_resultado)
    return df_resampleado


# ══════════════════════════════════════════════════════════════════════════════
# FASE 3 – Suavizado LOWESS sobre BIS
# ══════════════════════════════════════════════════════════════════════════════
def fase3_lowess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aplica LOWESS a la columna BIS con el parámetro de suavizado indicado
    en el artículo (frac = 0.03).
    """
    df = df.copy()
    x = df["t"].values.astype(float)
    y = df["BIS"].values.astype(float)

    # lowess devuelve array [y_suavizado] cuando return_sorted=False
    resultado_lowess = lowess(y, x, frac=LOWESS_FRAC, return_sorted=False)
    df["BIS"] = resultado_lowess
    return df


# ══════════════════════════════════════════════════════════════════════════════
# FASE 4 – Normalización de covariables estáticas
# ══════════════════════════════════════════════════════════════════════════════
def fase4_normalizacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza Edad, Peso, Altura con min-max.
    Codifica Sexo: F → 1, M → 0.
    Los valores normalizados se colocan en la primera fila (índice 0);
    el resto de la columna queda vacío (NaN),  donde
    las covariables estáticas se inyectan una sola vez.
    """
    df = df.copy()

    # Inicializamos las columnas normalizadas como NaN
    df["Edad_Norm"]   = np.nan
    df["Peso_norm"]   = np.nan
    df["Altura_norm"] = np.nan
    df["Sexo_norm"]   = np.nan

    # Tomamos los valores de la primera fila (son constantes para todo el paciente)
    edad_val   = df["Edad"].iloc[0]
    peso_val   = df["Peso"].iloc[0]
    altura_val = df["Altura"].iloc[0]
    sexo_val   = str(df["Sexo"].iloc[0]).strip().upper()

    # Normalización min-max
    edad_norm   = (edad_val   - EDAD_MIN)   / (EDAD_MAX   - EDAD_MIN)
    peso_norm   = (peso_val   - PESO_MIN)   / (PESO_MAX   - PESO_MIN)
    altura_norm = (altura_val - ALTURA_MIN) / (ALTURA_MAX - ALTURA_MIN)

    # Codificación de sexo
    if "F" in sexo_val:
        sexo_norm = 1
    elif "M" in sexo_val:
        sexo_norm = 0
    else:
        print(f"    [Fase 4] ADVERTENCIA: valor de Sexo no reconocido → '{sexo_val}'. Se asigna NaN.")
        sexo_norm = np.nan

    # Solo en la primera casilla (primera fila del dataframe re-muestreado)
    df.at[df.index[0], "Edad_Norm"]   = edad_norm
    df.at[df.index[0], "Peso_norm"]   = peso_norm
    df.at[df.index[0], "Altura_norm"] = altura_norm
    df.at[df.index[0], "Sexo_norm"]   = sexo_norm

    return df


# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN PRINCIPAL – Procesa todos los archivos CSV
# ══════════════════════════════════════════════════════════════════════════════
def procesar_paciente(ruta_csv: str, id_paciente: str) -> None:
    """Lee, limpia en 5 fases y guarda el archivo de un paciente."""

    print(f"\n{'─'*60}")
    print(f"  Procesando: Paciente_{id_paciente}.csv")
    print(f"{'─'*60}")

    # Lectura del archivo original
    df_crudo = pd.read_csv(ruta_csv)

    # Validación mínima de columnas esperadas
    columnas_requeridas = {"t", "PPF20_VOL", "RFTN20_VOL", "BIS",
                           "Edad", "Peso", "Altura", "Sexo"}
    columnas_faltantes = columnas_requeridas - set(df_crudo.columns)
    if columnas_faltantes:
        print(f"  ERROR: El archivo no tiene las columnas: {columnas_faltantes}. Se omite.")
        return

    # Aseguramos que el dataframe esté ordenado por tiempo
    df_crudo = df_crudo.sort_values("t").reset_index(drop=True)

    # ── Fase 1 ──────────────────────────────────────────
    print("  [Fase 1] Validación e interpolación...")
    df_f1 = fase1_validacion_interpolacion(df_crudo)

    # ── Fase 2 ──────────────────────────────────────────
    print(f"  [Fase 2] Re-muestreo a bloques de {TAMANO_BLOQUE}s...")
    df_f2 = fase2_remuestreo(df_f1)
    print(f"           Filas originales: {len(df_f1)} → Filas re-muestreadas: {len(df_f2)}")

    # ── Fase 3 ──────────────────────────────────────────
    print(f"  [Fase 3] Suavizado LOWESS (frac={LOWESS_FRAC}) sobre BIS...")
    df_f3 = fase3_lowess(df_f2)

    # ── Fase 4 ──────────────────────────────────────────
    print("  [Fase 4] Normalización de covariables estáticas...")
    df_f4 = fase4_normalizacion(df_f3)

    # ── Fase 5: Guardar ──────────────────────────────────
    nombre_salida = f"Paciente_{id_paciente}_limpio_ConFiltro.csv"
    ruta_salida   = os.path.join(CARPETA_SALIDA, nombre_salida)
    df_f4.to_csv(ruta_salida, index=False)
    print(f"  [Fase 5] Guardado → {ruta_salida}")


def main():
    # Crear carpeta de salida si no existe
    os.makedirs(CARPETA_SALIDA, exist_ok=True)

    # Listar todos los CSV con el patrón Paciente_<id>.csv
    patron = re.compile(r"^Paciente_(\d+)\.csv$", re.IGNORECASE)
    archivos = [f for f in os.listdir(CARPETA_ENTRADA) if patron.match(f)]

    if not archivos:
        print("No se encontraron archivos con el patrón 'Paciente_<id>.csv' en la carpeta de entrada.")
        return

    print(f"\nArchivos encontrados: {len(archivos)}")

    # Ordenar por id numérico para procesarlos en orden
    archivos.sort(key=lambda f: int(patron.match(f).group(1)))

    errores = []
    for nombre_archivo in archivos:
        id_paciente = patron.match(nombre_archivo).group(1)
        ruta_csv    = os.path.join(CARPETA_ENTRADA, nombre_archivo)
        try:
            procesar_paciente(ruta_csv, id_paciente)
        except Exception as e:
            print(f"  ERROR inesperado en {nombre_archivo}: {e}")
            errores.append(nombre_archivo)

    print(f"\n{'='*60}")
    print(f"  Procesamiento completado.")
    print(f"  Archivos procesados exitosamente : {len(archivos) - len(errores)}")
    if errores:
        print(f"  Archivos con error              : {len(errores)}")
        for e in errores:
            print(f"    - {e}")
    print(f"  Carpeta de salida: {CARPETA_SALIDA}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()


Archivos encontrados: 480

────────────────────────────────────────────────────────────
  Procesando: Paciente_17.csv
────────────────────────────────────────────────────────────
  [Fase 1] Validación e interpolación...
    [Fase 1] BIS: 537 valor(es) < 20 marcados como NaN
    [Fase 1] PPF20_VOL: 142 valor(es) decreciente(s) marcados como NaN
    [Fase 1] RFTN20_VOL: 124 valor(es) decreciente(s) marcados como NaN
    [Fase 1] BIS: 537 NaN interpolados (0 restantes tras interpolación)
    [Fase 1] PPF20_VOL: 246 NaN interpolados (0 restantes tras interpolación)
    [Fase 1] RFTN20_VOL: 226 NaN interpolados (0 restantes tras interpolación)
  [Fase 2] Re-muestreo a bloques de 10s...
           Filas originales: 19722 → Filas re-muestreadas: 1973
  [Fase 3] Suavizado LOWESS (frac=0.03) sobre BIS...
  [Fase 4] Normalización de covariables estáticas...
  [Fase 5] Guardado → C:\Users\yordi\Desktop\Entrenamiento_Limpios\Paciente_17_limpio_ConFiltro.csv

──────────────────────────────────────

## Limpieza en datos de Prueba y Validación.
Para los datos que el modelo usa para aprender y validar no se usa el filtro LOWESS

In [ ]:
"""
Script de limpieza de datos de anestesia - SIN filtro LOWESS
""""""""

Procesa dos conjuntos de datos en paralelo:
  - Prueba_Crudos     → Prueba_Limpios     (archivos: Paciente_{id}_limpio_SinFiltro.csv)
  - Validacion_Crudos → Validacion_Limpios (archivos: Paciente_{id}_limpio_SinFiltro.csv)

Fases aplicadas:
  1. Validación e interpolación de señales crudas
  2. Re-muestreo a bloques  (tasa de droga y promedio BIS)
  3. LOWESS omitido  para prueba y validación.
  4. Normalización de covariables estáticas (edad, peso, altura, sexo)
  5. Guardado de archivos limpios
"""

import os
import re
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
#  PARES DE CARPETAS A PROCESAR
#  Cada elemento: (carpeta_entrada, carpeta_salida)
# ─────────────────────────────────────────────
PARES_CARPETAS = [
    (
        r"C:\Users\yordi\Desktop\Prueba_Crudos",
        r"C:\Users\yordi\Desktop\Prueba_Limpios",
    ),
    (
        r"C:\Users\yordi\Desktop\Validacion_Crudos",
        r"C:\Users\yordi\Desktop\Validacion_Limpios",
    ),
]

# ─────────────────────────────────────────────
#  PARÁMETROS DE NORMALIZACIÓN (FASE 4)
# ─────────────────────────────────────────────
EDAD_MIN   = 16.00;   EDAD_MAX   = 90.00
PESO_MIN   = 34.10;   PESO_MAX   = 112.80
ALTURA_MIN = 140.20;  ALTURA_MAX = 186.60

# ─────────────────────────────────────────────
#  PARÁMETROS GENERALES
# ─────────────────────────────────────────────
TAMANO_BLOQUE = 10    # segundos por bloque 
BIS_MINIMO    = 20   # valores de BIS por debajo de este umbral → NaN (Fase 1)


# ══════════════════════════════════════════════════════════════════════════════
# FASE 1 – Validación e interpolación lineal
# ══════════════════════════════════════════════════════════════════════════════
def fase1_validacion_interpolacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    - BIS < 20  → NaN
    - PPF20_VOL y RFTN20_VOL: si un valor es menor que el anterior → NaN
      (se aplica de forma iterativa para cubrir caídas consecutivas)
    - Interpolación lineal sobre todos los NaN de las tres señales
    """
    df = df.copy()

    # --- BIS: marcar valores por debajo del mínimo clínico ---
    mascara_bis_invalido = df["BIS"] < BIS_MINIMO
    df.loc[mascara_bis_invalido, "BIS"] = np.nan
    n_bis = mascara_bis_invalido.sum()
    if n_bis:
        print(f"    [Fase 1] BIS: {n_bis} valor(es) < {BIS_MINIMO} marcados como NaN")

    # --- Volúmenes acumulados: deben ser no-decrecientes ---
    for columna_vol in ["PPF20_VOL", "RFTN20_VOL"]:
        serie = df[columna_vol].copy()
        n_invalidos = 0

        # Recorremos desde el segundo elemento
        for i in range(1, len(serie)):
            val_actual   = serie.iloc[i]
            val_anterior = serie.iloc[i - 1]

            # Si el anterior ya es NaN, comparamos con el último valor válido conocido
            if pd.isna(val_anterior):
                j = i - 1
                while j >= 0 and pd.isna(serie.iloc[j]):
                    j -= 1
                if j < 0:
                    continue    # no hay referencia válida, dejamos el valor
                val_anterior = serie.iloc[j]

            if not pd.isna(val_actual) and val_actual < val_anterior:
                serie.iloc[i] = np.nan
                n_invalidos += 1

        df[columna_vol] = serie
        if n_invalidos:
            print(f"    [Fase 1] {columna_vol}: {n_invalidos} valor(es) decreciente(s) marcados como NaN")

    # --- Interpolación lineal para rellenar todos los NaN ---
    for col in ["BIS", "PPF20_VOL", "RFTN20_VOL"]:
        n_nan_antes = df[col].isna().sum()
        df[col] = df[col].interpolate(method="linear", limit_direction="both")
        n_nan_despues = df[col].isna().sum()
        if n_nan_antes:
            print(f"    [Fase 1] {col}: {n_nan_antes} NaN interpolados "
                  f"({n_nan_despues} restantes tras interpolación)")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# FASE 2 – Re-muestreo a bloques 
# ══════════════════════════════════════════════════════════════════════════════
def fase2_remuestreo(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa las muestras en bloques de TAMANO_BLOQUE segundos.

    - PPF20_RATE  = (vol_final_bloque − vol_inicio_bloque) / duración_real
    - RFTN20_RATE = ídem para remifentanilo
    - BIS         = promedio del bloque

    El último bloque puede tener menos puntos que los anteriores.
    """
    filas_resultado = []
    n_filas = len(df)
    inicio  = 0

    while inicio < n_filas:
        fin    = min(inicio + TAMANO_BLOQUE, n_filas)   # índice exclusivo
        bloque = df.iloc[inicio:fin]

        # Tiempo del bloque (múltiplo de TAMANO_BLOQUE)
        t_bloque = (inicio // TAMANO_BLOQUE) * TAMANO_BLOQUE

        # Duración real del bloque (puede ser menor en el último)
        duracion_real = fin - inicio

        # Tasa propofol
        vol_inicio_ppf  = bloque["PPF20_VOL"].iloc[0]
        vol_fin_ppf     = bloque["PPF20_VOL"].iloc[-1]
        tasa_ppf        = (vol_fin_ppf - vol_inicio_ppf) / duracion_real

        # Tasa remifentanilo
        vol_inicio_rftn = bloque["RFTN20_VOL"].iloc[0]
        vol_fin_rftn    = bloque["RFTN20_VOL"].iloc[-1]
        tasa_rftn       = (vol_fin_rftn - vol_inicio_rftn) / duracion_real

        # Promedio BIS del bloque
        bis_promedio = bloque["BIS"].mean()

        # Covariables estáticas 
        edad   = bloque["Edad"].iloc[0]
        peso   = bloque["Peso"].iloc[0]
        altura = bloque["Altura"].iloc[0]
        sexo   = bloque["Sexo"].iloc[0]

        filas_resultado.append({
            "t"           : t_bloque,
            "PPF20_RATE"  : tasa_ppf,
            "RFTN20_RATE" : tasa_rftn,
            "BIS"         : bis_promedio,
            "Edad"        : edad,
            "Peso"        : peso,
            "Altura"      : altura,
            "Sexo"        : sexo,
        })

        inicio += TAMANO_BLOQUE

    df_resampleado = pd.DataFrame(filas_resultado)
    return df_resampleado


# ══════════════════════════════════════════════════════════════════════════════
# FASE 3 – OMITIDA: sin LOWESS para prueba y validación
# ══════════════════════════════════════════════════════════════════════════════
# El BIS de prueba y validación NO se suaviza para preservar la autenticidad
# de la señal.


# ══════════════════════════════════════════════════════════════════════════════
# FASE 4 – Normalización de covariables estáticas
# ══════════════════════════════════════════════════════════════════════════════
def fase4_normalizacion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza Edad, Peso, Altura con min-max usando los rangos fijos.
    Codifica Sexo: F → 1, M → 0.
    Los valores normalizados se colocan en la primera fila;
    el resto de la columna queda en NaN.
    """
    df = df.copy()

    # Inicializamos columnas normalizadas vacías
    df["Edad_Norm"]   = np.nan
    df["Peso_norm"]   = np.nan
    df["Altura_norm"] = np.nan
    df["Sexo_norm"]   = np.nan

    # Valores del paciente 
    edad_val   = df["Edad"].iloc[0]
    peso_val   = df["Peso"].iloc[0]
    altura_val = df["Altura"].iloc[0]
    sexo_val   = str(df["Sexo"].iloc[0]).strip().upper()

    # Normalización min-max
    edad_norm   = (edad_val   - EDAD_MIN)   / (EDAD_MAX   - EDAD_MIN)
    peso_norm   = (peso_val   - PESO_MIN)   / (PESO_MAX   - PESO_MIN)
    altura_norm = (altura_val - ALTURA_MIN) / (ALTURA_MAX - ALTURA_MIN)

    # Codificación binaria del sexo
    if "F" in sexo_val:
        sexo_norm = 1
    elif "M" in sexo_val:
        sexo_norm = 0
    else:
        print(f"    [Fase 4] ADVERTENCIA: valor de Sexo no reconocido → '{sexo_val}'. Se asigna NaN.")
        sexo_norm = np.nan

    
    df.at[df.index[0], "Edad_Norm"]   = edad_norm
    df.at[df.index[0], "Peso_norm"]   = peso_norm
    df.at[df.index[0], "Altura_norm"] = altura_norm
    df.at[df.index[0], "Sexo_norm"]   = sexo_norm

    return df


# ══════════════════════════════════════════════════════════════════════════════
# PROCESAMIENTO DE UN PACIENTE
# ══════════════════════════════════════════════════════════════════════════════
def procesar_paciente(ruta_csv: str, id_paciente: str, carpeta_salida: str) -> None:
    """Lee, limpia (sin LOWESS) y guarda el archivo de un paciente."""

    print(f"\n  {'─'*56}")
    print(f"  Procesando: Paciente_{id_paciente}.csv")
    print(f"  {'─'*56}")

    # Lectura del CSV original
    df_crudo = pd.read_csv(ruta_csv)

    # Validación de columnas requeridas
    columnas_requeridas = {"t", "PPF20_VOL", "RFTN20_VOL", "BIS",
                           "Edad", "Peso", "Altura", "Sexo"}
    columnas_faltantes = columnas_requeridas - set(df_crudo.columns)
    if columnas_faltantes:
        print(f"  ERROR: faltan columnas {columnas_faltantes}. Se omite el archivo.")
        return

    # Ordenar por tiempo 
    df_crudo = df_crudo.sort_values("t").reset_index(drop=True)

    # ── Fase 1 ──────────────────────────────────────────
    print("  [Fase 1] Validación e interpolación...")
    df_f1 = fase1_validacion_interpolacion(df_crudo)

    # ── Fase 2 ──────────────────────────────────────────
    print(f"  [Fase 2] Re-muestreo a bloques de {TAMANO_BLOQUE}s...")
    df_f2 = fase2_remuestreo(df_f1)
    print(f"           Filas originales: {len(df_f1)} → Filas re-muestreadas: {len(df_f2)}")

    # ── Fase 3: OMITIDA ──────────────────────────────────
    print("  [Fase 3] LOWESS omitido (conjunto de evaluación)")

    # ── Fase 4 ──────────────────────────────────────────
    print("  [Fase 4] Normalización de covariables estáticas...")
    df_f4 = fase4_normalizacion(df_f2)   # se aplica directamente sobre df_f2

    # ── Fase 5: Guardar ──────────────────────────────────
    nombre_salida = f"Paciente_{id_paciente}_limpio_SinFiltro.csv"
    ruta_salida   = os.path.join(carpeta_salida, nombre_salida)
    df_f4.to_csv(ruta_salida, index=False)
    print(f"  [Fase 5] Guardado → {ruta_salida}")


# ══════════════════════════════════════════════════════════════════════════════
# PROCESAMIENTO DE UNA CARPETA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
def procesar_carpeta(carpeta_entrada: str, carpeta_salida: str) -> None:
    """Procesa todos los CSV de una carpeta y los guarda en la carpeta de salida."""

    nombre_carpeta = os.path.basename(carpeta_entrada)
    print(f"\n{'═'*60}")
    print(f"  CARPETA: {nombre_carpeta}")
    print(f"  Entrada : {carpeta_entrada}")
    print(f"  Salida  : {carpeta_salida}")
    print(f"{'═'*60}")

    # Crear carpeta de salida si no existe
    os.makedirs(carpeta_salida, exist_ok=True)

    # Detectar archivos con el patrón Paciente_<id>.csv
    patron  = re.compile(r"^Paciente_(\d+)\.csv$", re.IGNORECASE)
    archivos = [f for f in os.listdir(carpeta_entrada) if patron.match(f)]

    if not archivos:
        print("  No se encontraron archivos con el patrón 'Paciente_<id>.csv'.")
        return

    print(f"  Archivos encontrados: {len(archivos)}")

    # Ordenar por id numérico
    archivos.sort(key=lambda f: int(patron.match(f).group(1)))

    errores = []
    for nombre_archivo in archivos:
        id_paciente = patron.match(nombre_archivo).group(1)
        ruta_csv    = os.path.join(carpeta_entrada, nombre_archivo)
        try:
            procesar_paciente(ruta_csv, id_paciente, carpeta_salida)
        except Exception as e:
            print(f"  ERROR inesperado en {nombre_archivo}: {e}")
            errores.append(nombre_archivo)

    # Resumen de la carpeta
    print(f"\n  Resumen '{nombre_carpeta}':")
    print(f"    Procesados correctamente : {len(archivos) - len(errores)}")
    if errores:
        print(f"    Con errores              : {len(errores)}")
        for e in errores:
            print(f"      - {e}")


# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print("\n" + "█"*60)
    print("  LIMPIEZA SIN LOWESS — Prueba y Validación")
    print("█"*60)

    # Procesar cada par de carpetas definido al inicio del script
    for carpeta_entrada, carpeta_salida in PARES_CARPETAS:
        procesar_carpeta(carpeta_entrada, carpeta_salida)

    print(f"\n{'█'*60}")
    print("  Procesamiento global completado.")
    print("█"*60 + "\n")


if __name__ == "__main__":
    main()


████████████████████████████████████████████████████████████
  LIMPIEZA SIN LOWESS — Prueba y Validación
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
  CARPETA: Prueba_Crudos
  Entrada : C:\Users\yordi\Desktop\Prueba_Crudos
  Salida  : C:\Users\yordi\Desktop\Prueba_Limpios
════════════════════════════════════════════════════════════
  Archivos encontrados: 128

  ────────────────────────────────────────────────────────
  Procesando: Paciente_16.csv
  ────────────────────────────────────────────────────────
  [Fase 1] Validación e interpolación...
    [Fase 1] BIS: 24 valor(es) < 20 marcados como NaN
    [Fase 1] PPF20_VOL: 40 valor(es) decreciente(s) marcados como NaN
    [Fase 1] RFTN20_VOL: 69 valor(es) decreciente(s) marcados como NaN
    [Fase 1] BIS: 24 NaN interpolados (0 restantes tras interpolación)
    [Fase 1] PPF20_VOL: 92 NaN interpolados (0 restantes tras interpolación)
    [Fase 1] RFTN20_VOL: 